# LLM pipeline smoke test

End-to-end check of `llm.py` against the real provider configured in `.env` (`settings.llm_provider` / `settings.model`). Makes real API calls -- requires `OPENAI_API_KEY` (or `ANTHROPIC_API_KEY`, depending on `settings.llm_provider`) to be set, and costs a small amount of real money (fractions of a cent per run at `gpt-4o-mini` rates).

Covers: a raw completion, `extract_sql` on the response, the full NL -> SQL -> execution pipeline against `data/olist.db`, cost/usage tracking, and the retry policy's fast-fail path on a non-retryable error.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import time
import llm
from db import execute_query

print("provider:", llm.settings.llm_provider, "| model:", llm.settings.model)

provider: openai | model: gpt-4o-mini


In [2]:
# raw completion + extract_sql
text = llm.complete(
    system="You are a SQLite expert. Reply with ONLY a single SQL SELECT statement, no prose, no markdown fences.",
    user="Write a query to count all rows in a table named orders.",
    max_tokens=100,
)
print("raw response:", repr(text))
print("extracted sql:", repr(llm.extract_sql(text)))

raw response: 'SELECT COUNT(*) FROM orders;'
extracted sql: 'SELECT COUNT(*) FROM orders'


In [3]:
# full pipeline: NL question -> generated SQL -> executed against the real db
text = llm.complete(
    system=(
        "You are a SQLite expert generating queries against this schema: "
        "orders(order_id, customer_id, order_status, order_purchase_timestamp). "
        "Reply with ONLY SQL."
    ),
    user="How many orders have order_status = delivered?",
    max_tokens=100,
)
sql = llm.extract_sql(text)
print("generated sql:", sql)

result = execute_query(sql)
print("execution result:", result.preview())

# cross-check against the known-good count from the schema notebook
expected = execute_query("SELECT count(*) FROM orders WHERE order_status = 'delivered'").rows[0][0]
assert result.ok and result.rows[0][0] == expected, "model's query didn't match the known-good count"
print(f"matches expected count ({expected})")

generated sql: SELECT COUNT(*) AS delivered_orders_count
FROM orders
WHERE order_status = 'delivered'
execution result: 1 rows | 96478
matches expected count (96478)


In [4]:
# cost / usage tracking
print("usage summary:", llm.usage_summary())
assert not llm.usage.snapshot().cost_incomplete, "pricing table is missing an entry for this model"

usage summary: {'calls': 2, 'input_tokens': 96, 'output_tokens': 29, 'cache_read_tokens': 0, 'cache_write_tokens': 0, 'cost_usd': 3.2e-05, 'cost_incomplete': False, 'mean_latency_ms': 1685.6, 'model': 'gpt-4o-mini', 'pricing_verified_on': '2026-08-05'}


In [5]:
# retry policy: a non-retryable error (unknown model -> 404) must fail fast
# on the first attempt, not burn through all max_attempts retries
start = time.perf_counter()
try:
    llm.complete(system="x", user="y", model="not-a-real-model-xyz", max_attempts=4)
    raise AssertionError("expected LLMError for a nonexistent model")
except llm.LLMError as e:
    elapsed = time.perf_counter() - start
    print(f"bad model failed in {elapsed:.1f}s (fast fail, not retried): {str(e)[:150]}")
    assert elapsed < 5, "a non-retryable error should fail on the first attempt, not retry"

bad model failed in 0.7s (fast fail, not retried): NotFoundError after 1 attempt(s): Error code: 404 - {'error': {'message': 'The model `not-a-real-model-xyz` does not exist or you do not have access t
